<a href="https://colab.research.google.com/github/Esbern/Sankey-diagrams/blob/main/KopiKopi_af_sankey2_policysource.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sankey diagram 2
### Policy source (skjult) → Target Group → Target → Land Use

#Import libaries

In [99]:
import requests
import pandas as pd
import plotly.graph_objects as go
import duckdb
import string
import numpy as np
import plotly.colors

#Adgang til Airtable

In [100]:
# Airtable credentials
api_key = 'patwjsizhgQyQkZkT.f9e8b1595df5b527d0d01d3a45af0dfa77eab63707e18398ad62f1f3818a9ce9'
base_id = 'apprKfEKZ2Ju74g9w'

In [101]:
# henter data fra Airtable via API og gemmer det i en pandas.DataFrame
def fetch_airtable_data(table_id):
    url = f"https://api.airtable.com/v0/{base_id}/{table_id}"
    headers = {"Authorization": f"Bearer {api_key}"}
    records = []
    params = {}

    while True:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code != 200:
            raise Exception(f"Failed to fetch data: {response.text}")
        data = response.json()
        records.extend([record["fields"] | {"id": record["id"]} for record in data["records"]])
        if "offset" in data:
            params["offset"] = data["offset"]
        else:
            break

    return pd.DataFrame(records)

In [102]:
# Fetch data
tabel0 = fetch_airtable_data("tblzHR1WHYHA5MlwQ")  # Policy Source
tabel1 = fetch_airtable_data("tbl7OYOXduME11uh7")  # Targets (mellemtabel)
tabel2 = fetch_airtable_data("tblVarbVYd96JUE6f")  # Target Group
tabel3 = fetch_airtable_data("tblTRyuT48bBN24QG")   # Slut-land uses

In [103]:
# Fjern rækker med manglende værdier i specifikke kolonner

# For tabel0: behold kun rækker hvor både 'Targets', 'Policy Source' og 'Targets (policy targets)' er udfyldt
tabel0 = tabel0.dropna(subset=['Policy source','Targets (policy targets)'])

# For tabel1: behold kun rækker hvor både 'Target name', 'Target Gruop' og 'Land uses' er udfyldt
tabel1 = tabel1.dropna(subset=['Target name','Target Group','Land uses'])

# For tabel2: behold kun rækker hvor både Target Group', 'Targets' er udfyldt
tabel2 = tabel2.dropna(subset=['Target Group', 'Targets'])

# For tabel3: behold kun rækker hvor både Target Group', 'Targets' er udfyldt
tabel3 = tabel3.dropna(subset=['Targets','Land conditions (from Targets)'])


In [104]:
# Explode og rename for videre behandling
t0 = tabel0.copy().explode("Targets (policy targets)").rename(columns={
    "Targets (policy targets)": "target_id"
})

t1 = tabel1.copy().explode("Target Group").rename(columns={
    "Target Group": "target_group_id"
})

t2 = tabel2.copy().explode("Targets").explode("Target Group").rename(columns={
    "Targets": "target_id",
    "Target Group": "target_group_id"
})

t3 = tabel3.copy().explode("Targets").rename(columns={
    "Targets": "target_id_tabel3"
})


In [105]:
# Filtrér direkte før registrering
allowed_policies = [
    "European Green Deal",
    "Common Agricultural Policy - Strategic plan 2023-2027",
    "EU biodiversity strategy 2030",
    "Aftale om et grønt Danmark (grøn trepart)",
    "Mere, bedre og større natur i Danmark",
    "Vandområdeplaner 2021-2027",
    "Technical Summary: Land Use and Climate Change",
    "Convention on Biological Diversity",
    "Common approach to integrating biodiversity and nature-based solutions for sustainable development into the United Nations policy and programme planning and delivery"
 ]
t0 = t0[t0["Policy source"].isin(allowed_policies)]

# allowed_groups = ["Climate", "Biodiversity", "Groundwater and water"]
# t2 = t2[t2["target_group_id"].isin(allowed_groups)]

In [106]:
# Register in DuckDB
duckdb.register("tabel0", t0)
duckdb.register("tabel1_exp", t1)
duckdb.register("tabel2_exp", t2)
duckdb.register("tabel3_exp", t3)

In [107]:
query = """
SELECT
    t0."Policy source"       AS policy_source,
    t1."Target name"         AS mid_target,
    t2."target_group_id"        AS target_group,
    t1."Land uses"           AS land_use,
    t3."Name"                AS final_target
FROM
    tabel0 t0
JOIN
    tabel1_exp t1 ON t0.target_id = t1.id
JOIN
    tabel2_exp t2 ON t1.target_group_id = t2.id
JOIN
    tabel3_exp t3 ON t2.target_id = t3.target_id_tabel3
"""
results = duckdb.sql(query).df()


In [108]:
# Forkort lange labels

results['target_group'] = results['target_group'].apply(lambda x: str(x[0]) if isinstance(x, (list, tuple, np.ndarray)) else str(x))
results['mid_target'] = results['mid_target'].apply(lambda x: str(x[0]) if isinstance(x, (list, tuple, np.ndarray)) else str(x))
results['land_use'] = results['land_use'].apply(lambda x: str(x[0]) if isinstance(x, (list, tuple, np.ndarray)) else str(x))

# Forkort lange labels
results['target_group'] = results['target_group'].apply(lambda x: x[:40] + '…' if len(x) > 40 else x)
results['mid_target'] = results['mid_target'].apply(lambda x: x[:40] + '…' if len(x) > 40 else x)
results['land_use'] = results['land_use'].apply(lambda x: x[:40] + '…' if len(x) > 40 else x)

# Definér labels til Sankey: Target Group → Target → Land uses
source_labels = results['target_group']
middle_labels = results['mid_target']
target_labels = results['land_use']

# Saml og opret unikke labels
all_labels = pd.concat([source_labels, middle_labels, target_labels])
unique_labels = pd.unique(all_labels)
label_to_index = {label: i for i, label in enumerate(unique_labels)}

# (Valgfrit) Print eksempel på mappings
for label in list(label_to_index.keys())[:5]:
    print(f"{label} → {label_to_index[label]}")



Klimasikring → 0
Friluftsinteresser og rekreative områder → 1
Nye former for organisering  → 2
Fødevaresikkerhed → 3
Økonomi samt regional og lokal udvikling → 4


In [109]:
# Forbindelser
links1 = pd.DataFrame({
    'source': source_labels.map(label_to_index),
    'target': middle_labels.map(label_to_index),
    'value': 1
})
links2 = pd.DataFrame({
    'source': middle_labels.map(label_to_index),
    'target': target_labels.map(label_to_index),
    'value': 1
})

all_links = pd.concat([links1, links2])

In [110]:
# Brug Plotlys palette
node_colors = plotly.colors.qualitative.Plotly

# Tildel farver til unikke labels (noder)
color_map = {label: node_colors[i % len(node_colors)] for i, label in enumerate(unique_labels)}

# Liste med farver i samme rækkefølge som unique_labels
node_colors_list = [color_map[label] for label in unique_labels]


# Funktion til at lysne farver (uden brug af gennemsigtighed)
def lighten(hex_color, factor=0.5):
    from plotly.colors import hex_to_rgb
    r, g, b = hex_to_rgb(hex_color)
    r = int(r + (255 - r) * factor)
    g = int(g + (255 - g) * factor)
    b = int(b + (255 - b) * factor)
    return f'rgb({r},{g},{b})'

# Lysnet farve til hver link ud fra source-node
link_colors = [lighten(node_colors_list[src], factor=0.8) for src in all_links['source']]



# Trin 3: Sankey-diagram
fig = go.Figure(data=[go.Sankey(
    arrangement='snap',
    node=dict(
        pad=60,
        thickness=20,
        line=dict(color="lightgray", width=0.5),
        label=list(unique_labels),
        color=node_colors_list
    ),
    link=dict(
        source=all_links['source'],
        target=all_links['target'],
        value=all_links['value'],
        color=link_colors
    )
)])
fig.update_layout(title_text="Target Group → Target →  Land uses", font_size=12, height=6000)
fig.show()